# *Nonlinear Arterial Hemodynamics*
## Chapter 1 companion — The Classical Picture of Arterial Hemodynamics

This notebook is the computational companion to Chapter 1. Its purpose is to reproduce the classical rigid-tube reference mechanics and to use VascuQuest to show how the chapter's principal dimensionless and harmonic ideas appear across a virtual arterial population.

The book nomenclature governs this notebook. Database-native field names appear only in the ingestion and mapping code.

The notebook is intentionally limited to the mechanics admitted by Chapter 1: incompressible Newtonian flow, scalar viscosity, a straight circular reference vessel, Poiseuille flow, linear pulsatility, harmonic representation, and wall shear traction. Constitutive anisotropy, geometry-sensitive modulation, wall compliance, and nonlinear synthesis are not developed here.

**Execution:** a clean Google Colab runtime should reproduce all results by choosing **Run all**. No cell requires a manual parameter choice.

### Chapter question

Chapter 1 asks what the classical Poiseuille–Womersley–WSS description resolves and where that information resides.

The notebook therefore has two equal tasks:

1. reconstruct the steady and pulsatile reference mechanics used by the chapter; and
2. use VascuQuest to expose how the governing scales vary across arterial sites and across the virtual population without changing the chapter's model assumptions.

The central distinction is preserved throughout: the velocity and pressure fields are volumetric quantities, whereas wall shear stress is a boundary traction derived from the resolved field.

### VascuQuest representation

VascuQuest is used here only as a source of validated PWDB virtual-population data needed to interrogate Chapter 1.

For each virtual subject, the notebook uses:

- age;
- heart rate;
- site pressure waveform;
- site flow-velocity waveform;
- site luminal-area waveform.

The database-native flow-velocity and luminal-area signals are combined internally to recover the book quantity $Q(t)$. The time-mean luminal area at each site is converted internally to the equivalent circular reference radius $R$. Neither database-native field names nor alternate symbols are introduced into the reader-facing analysis.

For each subject and site,

$$
\Omega=\frac{2\pi}{T},
\qquad
\alpha=R\sqrt{\frac{\Omega}{\nu}},
\qquad
\frac{\delta_W}{R}=\frac{\sqrt{2}}{\alpha}.
$$

The notebook uses the book reference values $\rho=1060\ {\rm kg\,m^{-3}}$ and $\mu=3.5\times10^{-3}\ {\rm Pa\,s}$, hence $\nu=\mu/\rho$.

The PWDB source age strata are retained as supplied rather than replaced by newly invented age bins. Age comparisons are descriptive only.

The source luminal-area waveform contains wall-motion information because PWDB is not a rigid-wall dataset. In this Chapter 1 notebook that waveform is used only to determine the reference radius and to reconstruct $Q(t)$. Wall compliance itself is not analysed; that mechanism belongs to Chapter 6.

In [ ]:
# Configuration and reproducibility constants
from pathlib import Path
import sys, json, hashlib, subprocess, zipfile

ROOT = Path("/content/nonlinear_arterial_hemodynamics_ch01")
FIG_DIR = ROOT / "figures"
DATA_DIR = ROOT / "data"
META_DIR = ROOT / "metadata"
for d in (ROOT, FIG_DIR, DATA_DIR, META_DIR):
    d.mkdir(parents=True, exist_ok=True)

VQ_REPOSITORY = "https://github.com/KNOWDYN/VascuQuest.git"
VQ_GIT_REF = "8307147d72e7a6f3ea3135895bd6f52927c67439"
PWDB_RECORD_ID = "3275625"
PWDB_DOI = "10.5281/zenodo.3275625"

# Book reference fluid properties.
rho = 1060.0       # kg m^-3
mu = 3.5e-3        # Pa s
nu = mu / rho      # m^2 s^-1

# Common arterial sites used to expose the range of Chapter 1 pulsatile scales.
SITES = [
    "AorticRoot", "ThorAorta", "AbdAorta", "Carotid",
    "Brachial", "Radial", "Femoral", "AntTibial",
]

print("Working directory:", ROOT)
print(f"nu = {nu:.6e} m^2/s")

In [ ]:
# Install the pinned VascuQuest revision used by this notebook.
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    f"git+{VQ_REPOSITORY}@{VQ_GIT_REF}"
])

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import vascuquest as vq

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("matplotlib:", matplotlib.__version__)
print("VascuQuest:", getattr(vq, "__version__", "version field not exposed"))

In [ ]:
# Acquire only the PWDB artifacts needed for Chapter 1.
# The model configurations provide subject metadata; the common-site waveform
# archive provides pressure, flow-velocity, and luminal-area waveforms.
ARTIFACTS = ["model_configurations", "common_site_waveforms_csv"]
verification = {}

for artifact in ARTIFACTS:
    subprocess.run(
        ["vascuquest", "dataset", "acquire",
         "--artifact", artifact, "--yes", "--format", "json"],
        check=True, text=True, capture_output=True,
    )
    verified = subprocess.run(
        ["vascuquest", "dataset", "verify",
         "--artifact", artifact, "--format", "json"],
        check=True, text=True, capture_output=True,
    )
    verification[artifact] = json.loads(verified.stdout)

status = subprocess.run(
    ["vascuquest", "dataset", "status", "--format", "json"],
    check=True, text=True, capture_output=True,
)
dataset_status = json.loads(status.stdout)
SOURCE_DIR = Path(dataset_status["managed_paths"]["source"])

(META_DIR / "artifact_verification.json").write_text(
    json.dumps(verification, indent=2), encoding="utf-8"
)
(META_DIR / "dataset_status.json").write_text(
    json.dumps(dataset_status, indent=2), encoding="utf-8"
)

print("Verified PWDB source:", SOURCE_DIR)

In [ ]:
# Open the verified dataset offline and establish deterministic subject metadata.
session = vq.open_dataset(source=SOURCE_DIR, offline=True)
assert session.identity.record_id == PWDB_RECORD_ID

age_result = session.get("age")
subject_ids = np.asarray(age_result.coordinates[0].values, dtype=str)
ages = np.asarray(age_result.values, dtype=float)

heart_rate_result = session.get("heart_rate", subjects=subject_ids.tolist())
heart_rate_ids = np.asarray(heart_rate_result.coordinates[0].values, dtype=str)
heart_rates = np.asarray(heart_rate_result.values, dtype=float)
assert np.array_equal(subject_ids, heart_rate_ids)

subject_meta = pd.DataFrame({
    "subject_id": subject_ids,
    "age_years": ages,
    "heart_rate_bpm": heart_rates,
})
subject_meta["subject_number"] = subject_meta["subject_id"].astype(int)
subject_meta = subject_meta.sort_values("subject_number").reset_index(drop=True)

# Deterministic representative subject:
# middle source age stratum, then median canonical subject number within it.
source_ages = sorted(subject_meta["age_years"].dropna().unique())
target_age = source_ages[len(source_ages) // 2]
age_stratum = subject_meta.loc[subject_meta["age_years"] == target_age]
age_stratum = age_stratum.sort_values("subject_number").reset_index(drop=True)
representative_subject = str(age_stratum.iloc[len(age_stratum) // 2]["subject_id"])

selection_record = {
    "rule": "middle source age stratum; median canonical subject number within that stratum",
    "source_age_years": [float(x) for x in source_ages],
    "representative_subject_id": representative_subject,
    "representative_age_years": float(target_age),
    "population_subject_count": int(len(subject_meta)),
}
(META_DIR / "subject_selection.json").write_text(
    json.dumps(selection_record, indent=2), encoding="utf-8"
)
display(pd.DataFrame([selection_record]))

In [ ]:
# Shared VascuQuest readers and the notebook-wide B&W plotting system.
WAVE_ZIP = SOURCE_DIR / "PWs_csv.zip"

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["DejaVu Serif"],
    "mathtext.fontset": "stix",
    "font.size": 9.0,
    "axes.labelsize": 9.0,
    "axes.titlesize": 9.5,
    "xtick.labelsize": 8.0,
    "ytick.labelsize": 8.0,
    "legend.fontsize": 7.8,
    "axes.linewidth": 0.75,
    "lines.linewidth": 1.2,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

BLACK, DARK, MID, LIGHT = "0.0", "0.28", "0.52", "0.74"

SITE_LABELS = {
    "AorticRoot": "Aortic root",
    "ThorAorta": "Thoracic aorta",
    "AbdAorta": "Abdominal aorta",
    "Carotid": "Carotid",
    "Brachial": "Brachial",
    "Radial": "Radial",
    "Femoral": "Femoral",
    "AntTibial": "Anterior tibial",
}

def clean_axes(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(False)

def save_figure(fig, stem):
    pdf = FIG_DIR / f"{stem}.pdf"
    png = FIG_DIR / f"{stem}.png"
    fig.savefig(pdf, bbox_inches="tight", pad_inches=0.03)
    fig.savefig(png, dpi=600, bbox_inches="tight", pad_inches=0.03)
    return pdf, png

def active_waveform(waveform):
    # VascuQuest waveforms can include trailing padding. Preserve only the
    # physically active prefix and reject internal missing values.
    y = np.asarray(waveform.values, dtype=float)
    t = np.asarray(waveform.time_coordinate.values, dtype=float)
    pad = (np.asarray(waveform.padding_mask, dtype=bool)
           if waveform.padding_mask is not None else np.zeros_like(y, dtype=bool))
    missing = (np.asarray(waveform.missing_mask, dtype=bool)
               if waveform.missing_mask is not None else np.zeros_like(y, dtype=bool))
    active = ~pad
    last = np.flatnonzero(active)[-1] + 1
    y, t, missing = y[:last], t[:last], missing[:last]
    if np.any(missing) or np.any(~np.isfinite(y)):
        raise ValueError("Waveform contains an internal missing sample.")
    return t, y

def representative_p_and_Q(subject_id, site_id="AorticRoot"):
    # Database-native signal names are confined to this mapping layer.
    site = vq.MeasurementSite(site_id)
    pressure = session.waveform("pressure", subject=subject_id, location=site)
    flow_velocity = session.waveform("flow_velocity", subject=subject_id, location=site)
    luminal_area = session.waveform("luminal_area", subject=subject_id, location=site)

    tp, p_values = active_waveform(pressure)
    tu, velocity_values = active_waveform(flow_velocity)
    ta, area_values = active_waveform(luminal_area)

    n = min(len(p_values), len(velocity_values), len(area_values))
    p_values = p_values[:n]
    Q_values = velocity_values[:n] * area_values[:n]
    phase = np.arange(n, dtype=float) / n

    return pd.DataFrame({
        "phase": phase,
        "p_mmHg": p_values,
        "Q_m3_s": Q_values,
    })

def _wave_member_name(site_id, source_signal):
    # Efficient bulk access to the same checksum-verified PWDB waveform archive.
    basename = f"PWs_{site_id}_{source_signal}.csv"
    with zipfile.ZipFile(WAVE_ZIP, "r") as zf:
        matches = [name for name in zf.namelist() if Path(name).name == basename]
    if len(matches) != 1:
        raise RuntimeError(f"Expected one {basename!r}; found {len(matches)}")
    return matches[0]

def load_waveform_matrix(site_id, source_signal):
    member = _wave_member_name(site_id, source_signal)
    with zipfile.ZipFile(WAVE_ZIP, "r") as zf:
        with zf.open(member, "r") as raw:
            frame = pd.read_csv(raw, low_memory=False)
    ids = np.asarray([str(int(x)) for x in frame.iloc[:, 0].to_numpy()], dtype=str)
    values = frame.iloc[:, 1:].to_numpy(dtype=float)
    return ids, values

def first_harmonic_coefficients(y):
    # Chapter 1 uses a linear Fourier representation; this helper computes the
    # ordinary discrete coefficients of one complete periodic waveform.
    y = np.asarray(y, dtype=float)
    return np.fft.rfft(y) / len(y)

print("Shared helpers ready.")

# Governing mechanics before reduction

For an incompressible Newtonian fluid,

$$
\nabla\cdot\mathbf u=0,
$$

and

$$
\rho\left(
\frac{\partial\mathbf u}{\partial t}
+(\mathbf u\cdot\nabla)\mathbf u
\right)
=-\nabla p+\mu\nabla^2\mathbf u.
$$

The notebook does not replace the continuum-mechanical development of Appendices C and D. Its role is to make visible what is retained after the Chapter 1 kinematic restrictions are imposed.

The surface traction is

$$
\mathbf t(\mathbf n)=\boldsymbol\sigma\mathbf n.
$$

This distinction between a volume equation and a surface traction is retained throughout the computational analysis.

# The steady reference: Poiseuille flow

Under the Chapter 1 restrictions,

$$
\mathbf u=u_z(r)\mathbf e_z,
\qquad
G=-\frac{dp}{dz},
$$

and

$$
u_z(r)=\frac{G}{4\mu}\left(R^2-r^2\right).
$$

The corresponding shear stress is

$$
\tau_{rz}(r)=-\frac{G}{2}r,
$$

with

$$
Q=\frac{\pi R^4}{8\mu}G,
\qquad
|\tau_w|=\frac{GR}{2}
=\frac{4\mu Q}{\pi R^3}.
$$

Using the book-wide normalized radius $x=r/R$, the following calculation reproduces the field structure without choosing an arbitrary vessel size or forcing magnitude.

In [ ]:
# Deterministic Poiseuille reconstruction.
# The cylindrical derivative and radial regularity correspond to the
# cylindrical-coordinate machinery summarized in Appendix B.
x = np.linspace(0.0, 1.0, 500)

# Normalized directly by the centerline and wall values; no new symbol is needed.
velocity_ratio = 1.0 - x**2
shear_ratio = x

fig, ax = plt.subplots(figsize=(5.6, 3.25))
ax.plot(x, velocity_ratio, color=BLACK, linestyle="-",
        label=r"$u_z(r)/u_z(0)$")
ax.plot(x, shear_ratio, color=DARK, linestyle="--",
        label=r"$\tau_{rz}(r)/\tau_{rz}(R)$")
ax.set_xlabel(r"Normalized radius, $x=r/R$")
ax.set_ylabel("Normalized value")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.04)
ax.legend(frameon=False)
clean_axes(ax)
fig.tight_layout()

save_figure(fig, "ch01_poiseuille_reference")
plt.show()

The velocity field carries information throughout the cross-section, while the wall shear value is obtained only at $x=1$. In the Poiseuille reference problem the wall traction is exact, but it is still a boundary projection of a richer interior solution.

This reconstruction is a deterministic consequence of the Chapter 1 equations; no VascuQuest data are involved.

# Dimensionless balances and pulsatility

## What the Reynolds number measures

The Reynolds number is

$$
\mathrm{Re}=\frac{\rho U_0L_0}{\mu}
=\frac{U_0L_0}{\nu}.
$$

It compares convective inertia with viscous diffusion. Chapter 1 emphasizes that this does **not** mean that a nonzero or appreciable Reynolds number forces convective acceleration to appear in the fully developed Womersley reference state. Under the imposed kinematics,

$$
(\mathbf u\cdot\nabla)\mathbf u=\mathbf 0
$$

identically.

For that reason, this notebook does not construct a population Reynolds-number ranking from an arbitrary choice of $U_0$ and $L_0$. Such a plot would add a convention that is not needed to understand the Chapter 1 mechanism.

## Introducing pulsatility and the Womersley number

For the rigid fully developed unsteady state,

$$
\rho\frac{\partial u_z}{\partial t}
=
G(t)
+
\mu\frac{1}{r}\frac{\partial}{\partial r}
\left(
r\frac{\partial u_z}{\partial r}
\right).
$$

The unsteady-to-viscous balance is measured by

$$
\alpha=R\sqrt{\frac{\Omega}{\nu}},
$$

and the oscillatory viscous penetration thickness satisfies

$$
\frac{\delta_W}{R}=\frac{\sqrt{2}}{\alpha}.
$$

The complete Bessel-function Womersley solution belongs to Chapter 3. Chapter 1 needs only the balance represented by $\alpha$ and the physical consequence for radial viscous penetration.

In [ ]:
# Population map of the Chapter 1 Womersley scale.
# R is obtained from the time-mean luminal area but wall motion is not analysed.
rows = []
meta = subject_meta.set_index("subject_id")

for site in SITES:
    print("Processing", SITE_LABELS[site])
    ids, area_matrix = load_waveform_matrix(site, "A")

    # Time mean is taken over finite samples only; PWDB trailing padding is NaN.
    mean_area = np.nanmean(area_matrix, axis=1)
    R_values = np.sqrt(mean_area / np.pi)

    for sid, R_value in zip(ids, R_values):
        if sid not in meta.index or not np.isfinite(R_value):
            continue
        heart_rate_bpm = float(meta.loc[sid, "heart_rate_bpm"])
        age_years = float(meta.loc[sid, "age_years"])

        # T and Omega implement the Chapter 1 periodic forcing scale.
        T_value = 60.0 / heart_rate_bpm
        Omega_value = 2.0 * np.pi / T_value
        alpha_value = R_value * np.sqrt(Omega_value / nu)
        penetration_ratio = np.sqrt(2.0) / alpha_value

        rows.append({
            "subject_id": sid,
            "age_years": age_years,
            "site": site,
            "R_m": R_value,
            "T_s": T_value,
            "Omega_s_inv": Omega_value,
            "alpha": alpha_value,
            "deltaW_over_R": penetration_ratio,
        })

alpha_df = pd.DataFrame(rows)
alpha_df.to_csv(DATA_DIR / "ch01_population_womersley.csv", index=False)

# Panel (a): site-by-site population range.
groups = [
    alpha_df.loc[alpha_df["site"] == site, "alpha"].dropna().to_numpy()
    for site in SITES
]

# Panel (b): retain PWDB source age strata exactly as supplied.
selected_age_sites = ["AorticRoot", "Carotid", "Femoral", "Radial"]
age_summary = (
    alpha_df.loc[alpha_df["site"].isin(selected_age_sites)]
    .groupby(["age_years", "site"])["alpha"]
    .median()
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.3))

axes[0].boxplot(
    groups,
    labels=[SITE_LABELS[s] for s in SITES],
    showfliers=False,
    whis=(5, 95),
    widths=0.58,
    medianprops={"color": BLACK, "linewidth": 1.3},
    boxprops={"color": BLACK, "linewidth": 0.9},
    whiskerprops={"color": DARK, "linewidth": 0.8},
    capprops={"color": DARK, "linewidth": 0.8},
)
axes[0].set_ylabel(r"Womersley number, $\alpha$")
axes[0].tick_params(axis="x", rotation=52)
clean_axes(axes[0])

styles = [
    (BLACK, "-", "o"),
    (DARK, "--", "s"),
    (MID, "-.", "^"),
    (LIGHT, ":", "D"),
]
for site, (gray, ls, marker) in zip(selected_age_sites, styles):
    sub = age_summary.loc[age_summary["site"] == site]
    axes[1].plot(
        sub["age_years"], sub["alpha"],
        color=gray, linestyle=ls, marker=marker, markersize=4,
        label=SITE_LABELS[site],
    )

axes[1].set_xlabel("PWDB source age (years)")
axes[1].set_ylabel(r"Median $\alpha$")
axes[1].legend(frameon=False)
clean_axes(axes[1])

fig.tight_layout(w_pad=1.3)
save_figure(fig, "ch01_womersley_population")
plt.show()

display(
    alpha_df.groupby("site")["alpha"]
    .agg(["count", "median", "min", "max"])
    .reindex(SITES)
    .rename(index=SITE_LABELS)
)

The population plot is an **observation in VascuQuest**, not a new analytical law. It shows that the balance represented by $\alpha$ is not one fixed arterial number: it changes with reference radius and forcing period, and therefore varies across sites and across the virtual population.

The age-stratified panel is descriptive. It should be read only as variation within PWDB under its virtual-population construction. It does not establish that age causes a particular change in $\alpha$.

The Chapter 1 interpretation remains the analytical relation

$$
\frac{\delta_W}{R}=\frac{\sqrt{2}}{\alpha}.
$$

Larger $\alpha$ corresponds to a smaller fraction of the radius occupied by the oscillatory viscous penetration scale.

# Harmonic representation of a periodic waveform

Chapter 1 represents a periodic forcing as

$$
G(t)=G_0+\Re\left\{
\sum_{m=1}^{M}\widehat G_m e^{im\Omega t}
\right\},
$$

with the corresponding linear velocity reconstruction

$$
u_z(r,t)=u_0(r)+\Re\left\{
\sum_{m=1}^{M}\widehat u_m(r)e^{im\Omega t}
\right\}.
$$

Each harmonic has

$$
\Omega_m=m\Omega,
\qquad
\alpha_m=\sqrt{m}\,\alpha.
$$

VascuQuest is used here to show how a real periodic arterial waveform populates those harmonics. The source waveform is not treated as proof of the rigid-tube model; it is used only as a physically structured periodic signal to interrogate the chapter's linear harmonic representation.

In [ ]:
# Representative aortic-root pressure and flow waveforms.
# Q(t) is reconstructed only in the VascuQuest mapping layer from the source
# flow-velocity and luminal-area signals, then carried forward as the book quantity Q.
wave = representative_p_and_Q(representative_subject, "AorticRoot")
wave["Q_ml_s"] = wave["Q_m3_s"] * 1e6
wave.to_csv(DATA_DIR / "ch01_representative_p_Q.csv", index=False)

fig, axes = plt.subplots(2, 1, figsize=(6.2, 4.2), sharex=True)

axes[0].plot(wave["phase"], wave["p_mmHg"], color=BLACK)
axes[0].set_ylabel(r"$p$ (mmHg)")
clean_axes(axes[0])

axes[1].plot(wave["phase"], wave["Q_ml_s"], color=BLACK)
axes[1].set_ylabel(r"$Q$ (mL s$^{-1}$)")
axes[1].set_xlabel(r"Normalized time, $t/T$")
clean_axes(axes[1])

fig.tight_layout(h_pad=0.5)
save_figure(fig, "ch01_representative_waveforms")
plt.show()

In [ ]:
# Fourier decomposition of the representative Q(t) waveform.
# This is the discrete counterpart of the Chapter 1 linear harmonic expansion.
Q_values = wave["Q_ml_s"].to_numpy()
coeff = first_harmonic_coefficients(Q_values)

M = min(10, len(coeff) - 1)
m = np.arange(1, M + 1)
amplitude = 2.0 * np.abs(coeff[m])
relative_amplitude = amplitude / amplitude[0]

# Use the representative subject's heart rate and time-mean aortic-root area
# to evaluate the Chapter 1 alpha_m = sqrt(m) alpha scaling.
area_wave = session.waveform(
    "luminal_area",
    subject=representative_subject,
    location=vq.MeasurementSite("AorticRoot"),
)
_, area_values = active_waveform(area_wave)
R_rep = np.sqrt(np.mean(area_values) / np.pi)

heart_rate_rep = float(
    subject_meta.loc[
        subject_meta["subject_id"] == representative_subject,
        "heart_rate_bpm"
    ].iloc[0]
)
T_rep = 60.0 / heart_rate_rep
Omega_rep = 2.0 * np.pi / T_rep
alpha_rep = R_rep * np.sqrt(Omega_rep / nu)

alpha_m = np.sqrt(m) * alpha_rep
penetration_m = np.sqrt(2.0) / alpha_m

harmonic_df = pd.DataFrame({
    "m": m,
    "Q_amplitude_relative_to_m1": relative_amplitude,
    "alpha_m": alpha_m,
    "deltaW_m_over_R": penetration_m,
})
harmonic_df.to_csv(DATA_DIR / "ch01_harmonic_scaling.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(7.0, 3.0))

axes[0].bar(
    m, relative_amplitude, width=0.68,
    facecolor="white", edgecolor=BLACK, linewidth=0.9
)
axes[0].set_xlabel(r"Harmonic number, $m$")
axes[0].set_ylabel(r"$|\widehat Q_m|/|\widehat Q_1|$")
axes[0].set_xticks(m)
clean_axes(axes[0])

axes[1].plot(
    m, penetration_m,
    color=BLACK, linestyle="-", marker="o", markersize=4
)
axes[1].set_xlabel(r"Harmonic number, $m$")
axes[1].set_ylabel(r"$\delta_{W,m}/R$")
axes[1].set_xticks(m)
clean_axes(axes[1])

fig.tight_layout(w_pad=1.4)
save_figure(fig, "ch01_harmonic_content_and_penetration")
plt.show()

display(harmonic_df)

The left panel is an **observation in the selected VascuQuest waveform**: different harmonics carry different fractions of the periodic flow signal.

The right panel is a **computed consequence of the Chapter 1 scaling**:

$$
\alpha_m=\sqrt{m}\,\alpha,
\qquad
\frac{\delta_{W,m}}{R}
=
\frac{\sqrt{2}}{\alpha_m}.
$$

Thus higher harmonic number does not merely add temporal detail. Within the classical linear model it also changes the radial unsteady-viscous balance. Chapter 3 develops the corresponding harmonic Womersley transfer functions in full.

# Wall shear stress as a traction

For a surface with unit normal $\mathbf n$,

$$
\mathbf t(\mathbf n)=\boldsymbol\sigma\mathbf n.
$$

Its tangential part is

$$
\mathbf t_t
=
(\mathbf I-\mathbf n\mathbf n)\mathbf t,
$$

and for the Newtonian fluid

$$
\boldsymbol\tau_w
=
(\mathbf I-\mathbf n\mathbf n)
(2\mu\mathbf D)\mathbf n.
$$

For the straight-tube Chapter 1 state,

$$
\boldsymbol\tau_w
=
\mu
\left.
\frac{\partial u_z}{\partial r}
\right|_{R}
\mathbf e_z
$$

for the stated traction orientation.

The Poiseuille reconstruction above already shows the computational meaning of this definition: the interior radial derivative produces a shear field, and the wall value is obtained by evaluating that field at $r=R$.

No VascuQuest wall-shear surrogate is introduced here. Doing so from a generic pulsatile $Q(t)$ waveform would require an additional profile assumption or the full Womersley transfer relation. The latter belongs to Chapter 3. This notebook therefore preserves the mechanical definition rather than manufacturing a quantity that Chapter 1 has not yet solved.

# What the classical description actually resolves

Within the strict Chapter 1 reference problem,

$$
u_r=0,
\qquad
u_\theta=0,
\qquad
\frac{\partial}{\partial\theta}=0,
\qquad
\frac{\partial u_z}{\partial z}=0,
\qquad
r_{\mathrm{wall}}=R.
$$

The computational results above reinforce the chapter's division of information:

| Quantity or relation | Status in Chapter 1 |
|---|---|
| $u_z(r)$ | resolved interior field in the steady reference |
| $Q$ | cross-sectional integral of the resolved velocity field |
| $\tau_{rz}(r)$ | interior shear-stress component derived from the velocity gradient |
| $\boldsymbol\tau_w$ | boundary traction obtained from the stress field |
| $\alpha$ | dimensionless unsteady-to-viscous balance |
| $\delta_W/R$ | penetration-scale consequence of $\alpha$ |
| harmonic content | admitted by linear periodic representation |
| constitutive anisotropy | withheld |
| geometry-sensitive modulation | withheld |
| wall compliance | withheld |
| nonlinear harmonic transfer | withheld |

VascuQuest adds population context to the scales and waveforms, but it does not alter this model boundary.

# What the reader should learn

1. **Poiseuille flow is a field solution, not merely a resistance formula.** Its velocity, shear, flow rate, and wall traction are different quantities generated by the same restricted mechanics.

2. **WSS is mechanically exact within the reference problem but remains a boundary quantity.** It is obtained from the tangential projection of the Cauchy traction after the velocity field is known.

3. **Reynolds number and Womersley number organize different balances.** The fully developed Womersley state can have a nontrivial $\mathrm{Re}$ while its convective acceleration remains identically zero.

4. **The Womersley scale varies materially across the virtual arterial population.** This is a VascuQuest observation; the governing physical interpretation still comes from the analytical definition of $\alpha$.

5. **A periodic arterial waveform naturally contains multiple harmonics.** In the linear classical model, harmonic $m$ is not merely another Fourier coefficient: it also has $\alpha_m=\sqrt{m}\alpha$ and therefore a distinct viscous penetration scale.

6. **The classical reference obtains its clarity by exclusion.** The mechanisms withheld by Chapter 1 should not be smuggled back into its notebook analysis.

# Chapter-enrichment candidates

The notebook produces four candidate visual results.

**Candidate 1 — normalized Poiseuille velocity and shear profiles.**  
Strong book candidate because it makes the field-versus-boundary distinction immediately visible with no dependence on population data.

**Candidate 2 — representative $p(t)$ and $Q(t)$ waveforms.**  
Useful primarily as notebook context. Promotion to the book is optional because the chapter can remain complete without a representative virtual subject.

**Candidate 3 — population distribution of $\alpha$ with source-age comparison.**  
Potential book candidate if the chapter would benefit from showing that the classical pulsatile balance spans a broad arterial range. Any caption must state clearly that this is a PWDB virtual-population observation.

**Candidate 4 — harmonic content with $\delta_{W,m}/R$.**  
Potentially valuable if the chapter needs a stronger visual bridge between periodic waveforms and the $\alpha_m=\sqrt{m}\alpha$ scaling. It should not duplicate the fuller Womersley treatment in Chapter 3.

No figure is promoted automatically. The book remains complete without this notebook.

In [ ]:
# Reproducibility record and execution manifest.
manifest = {
    "book": "Nonlinear Arterial Hemodynamics",
    "chapter": 1,
    "chapter_title": "The Classical Picture of Arterial Hemodynamics",
    "vascuquest_git_ref": VQ_GIT_REF,
    "pwdb_record_id": PWDB_RECORD_ID,
    "pwdb_doi": PWDB_DOI,
    "rho_kg_m3": rho,
    "mu_Pa_s": mu,
    "nu_m2_s": nu,
    "representative_subject_id": representative_subject,
    "representative_age_years": float(target_age),
    "sites": SITES,
    "source_age_strata_years": [float(x) for x in source_ages],
    "python": sys.version.split()[0],
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "matplotlib": matplotlib.__version__,
    "execution_status": "completed to this cell",
}

(META_DIR / "reproducibility_manifest.json").write_text(
    json.dumps(manifest, indent=2), encoding="utf-8"
)

print(json.dumps(manifest, indent=2))
print("\nGenerated figures:")
for path in sorted(FIG_DIR.glob("*.pdf")):
    print(" -", path.name)

# Reproducibility record

A successful **Run all** execution writes:

- publication-quality PDF and PNG figures;
- the population and representative-case data used by those figures;
- VascuQuest/PWDB verification metadata;
- the deterministic representative-subject record;
- a final reproducibility manifest.

The notebook deliberately contains no interactive branch, widget, or manual parameter selection. Its scientific results are generated deterministically from the pinned VascuQuest revision and the verified PWDB source.